## Building the Allegheny County polluter point dataset


2. **Download the TRI file**
   - Source: EPA TRI Basic Data Files page (epa.gov/toxics-release-inventory-tri-program)
   - Select your year and Pennsylvania, then download the CSV.
   - Also download the "Guide to Using the TRI Basic Data Files" PDF. Column names include number prefixes that have changed across years.
   - Each row is one facility-chemical combination, not one facility.

7. **Build and check geometry**
   - Create point geometry from longitude (x) and latitude (y), in that order.
   - Set CRS to EPSG:4326, then reproject to EPSG:2272.
   - Plot over the Allegheny County boundary: every point should fall inside the county, and Clairton should land on the river in Clairton.
   - Fix or drop facilities with missing or wrong coordinates, and document it.

8. **Save and hand off**
   - Save as GeoPackage or GeoJSON in EPSG:2272.
   - Columns: facility ID, name, city, parent company, total air (lbs), RSEI Score, RSEI Hazard, rank, top-10 flag, U.S. Steel flag.
   - Record the year, selection rule, and join results for the report's data section.

## Step 1: Get the Allegheny County shapefile

In [1]:
# import allegheny county shapefile from census python package
from dotenv import load_dotenv
import os
import census as c
from us import states
import pygris
import geopandas as gpd

# load_dotenv()
# API_KEY = os.getenv("CENSUS_API_KEY")

# alleg = pygris.counties(states.PA.fips).query("NAME == 'Allegheny'").to_crs("EPSG:2272")

# # save alleg as geojson to disk
# alleg.to_file("alleg.geojson", driver="GeoJSON")

# load allegh from geojson file
alleg = gpd.read_file("alleg.geojson")

## Step 2: Process the TRI and RSEI data, select the most polluting facilities

In [2]:
import pandas as pd

tri_basic=pd.read_csv("tri_2024_pa.csv")
# allegh=tri_basic[tri_basic['7. COUNTY']=='ALLEGHENY']

tri_allegh=(
    tri_basic
    .query("`7. COUNTY` == 'ALLEGHENY'")
    .filter(items=[
        "2. TRIFD", "4. FACILITY NAME", "5. STREET ADDRESS", "6. CITY",
        "12. LATITUDE", "13. LONGITUDE", "15. PARENT CO NAME",
        "50. UNIT OF MEASURE", "51. 5.1 - FUGITIVE AIR", "52. 5.2 - STACK AIR"
    ])
    .set_axis(["trifd", "name", "address", "city", "lat", "lon", "parent",
               "unit", "fugitive_air", "stack_air"], axis=1)
    .query("unit == 'Pounds'")
    .assign(air_lbs=lambda d: d["fugitive_air"].fillna(0) + d["stack_air"].fillna(0))
    .groupby("trifd").agg(
        name=("name", "first"),
        address=("address", "first"),
        city=("city", "first"),
        parent=("parent", "first"),
        lat=("lat", "first"),
        lon=("lon", "first"),
        air_lbs=("air_lbs", "sum"),
    )
    .reset_index()
    .sort_values("air_lbs", 
                 ascending=False)
    .reset_index(drop=True)
    .assign(rank=lambda d: d.index + 1)
)
tri_allegh

,trifd,name,address,city,parent,lat,lon,air_lbs,rank
0,15025SSCLR400ST,USS-CLAIRTON PLANT,400 STATE ST MS 71,CLAIRTON,US STEEL CORP,40.294941,-79.874537,775872.720,1
1,15104SSDGRBRADD,USS MON VALLEY WORKS - EDGAR THOMSON PLANT,BRADDOCK AVE & 13TH ST,BRADDOCK,US STEEL CORP,40.397262,-79.859713,96883.690,2
2,15034SSRVNPOBOX,USS MON VALLEY WORKS - IRVIN PLANT,CAMP HOLLOW RD,WEST MIFFLIN,US STEEL CORP,40.340524,-79.914952,73871.600,3
3,15088HRCLSSTATE,SYNTHOMER JEFFERSON HILLS LLC,2200 STATE HWY 837,JEFFERSON HILLS,YULE CATTO INC.,40.267336,-79.903751,50394.000,4
4,15122LBRTY1575L,LIBERTY PULTRUSIONS,1575 LEBANON SCHOOL RD,WEST MIFFLIN,NaN,40.338887,-79.899186,28824.910,5
5,15104MSDVT13BRA,TMS INTERNATIONAL LLC,1300 BRADDOCK AVE,BRADDOCK,TMS INTERNATIONAL CORP,40.397191,-79.859663,27877.980,6
6,15144PPGND125CO,PPG INDUSTRIESINC-SPRINGDALE COMPLEX,125 COLFAX ST,SPRINGDALE,PPG INDUSTRIES INC,40.537373,-79.784005,22186.090,7
7,15225NVLLC2800N,NEVILLE CHEMICAL CO,2800 NEVILLE RD,PITTSBURGH,NaN,40.500626,-80.098238,14392.000,8
8,15225SHLND2650N,INEOS COMPOSITES US LLC,2650 NEVILLE RD,PITTSBURGH,INEOS ENTERPRISES US HOLDCO LLC,40.497715,-80.091249,13776.000,9
9,15225CLGNC200NE,CALGON CARBON CORP NEVILLE ISLAND PLANT,200 NEVILLE RD,PITTSBURGH,KURARAY HOLDINGS USA INC,40.492203,-80.079813,6999.000,10


In [3]:
easyrsei = (
    pd.read_excel("easyrsei_2022_air.xlsx")
    .rename(
        columns={
            "Facility Name": "name",
            "Street": "address",
            "RSEI Score": "rsei_score",
            "TRI Pounds": "tri_pounds"
        })
    .filter(items=[
        "name", "address", "rsei_score", "tri_pounds"]
        )
)
easyrsei

,name,address,rsei_score,tri_pounds
0,USS-CLAIRTON PLANT,400 STATE ST MS 71,83152.966207,9.251498e+06
1,USS MON VALLEY WORKS - EDGAR THOMSON PLANT,BRADDOCK AVE & 13TH ST,5536.456060,3.311955e+06
2,LIBERTAS COPPER LLC,100 WASHINGTON ST,745.215200,2.682415e+06
3,ATI FLAT ROLLED PRODUCTS HOLDINGS LLC,100 RIVER RD,12838.631711,2.364991e+06
4,SYNTHOMER JEFFERSON HILLS LLC,2200 STATE HWY 837,32.298782,1.879081e+06
5,PPG INDUSTRIESINC-SPRINGDALE COMPLEX,125 COLFAX ST,77704.927905,7.597395e+05
6,MCCONWAY & TORLEY LLC PLT 1801,109 48TH ST,12867.228911,5.010040e+05
7,UNIVERSAL STAINLESS & ALLOY PRODUCTS INC,600 MAYER ST,47409.307944,3.384130e+05
8,HOLTEC MANUFACTURING,750 BRADDOCK AVE,1750.380808,2.828373e+05
9,MATTHEWS INTERNATIONAL CORP,1315 W LIBERTY AVE,1455.599460,2.767640e+05


In [4]:
import mapclassify as mc
import numpy as np

def top_class(values, k):
    """True for rows in the highest Jenks class (log10 scale). Zeros are never top."""
    out = pd.Series(False, index=values.index)
    pos = values[values > 0]
    jenks = mc.FisherJenks(np.log10(pos), k=k)
    out[pos.index] = jenks.yb == k - 1
    return out
 
tri_allegh["tri_top"] = top_class(tri_allegh["air_lbs"], k=5)
easyrsei["rsei_top"] = top_class(easyrsei["rsei_score"], k=4)

/var/folders/92/rrvsc6_x7slf490khsx_xn2c0000gn/T/ipykernel_59244/760923391.py:12: UserWarning: Numba not installed. Using slow pure python version.
  tri_allegh["tri_top"] = top_class(tri_allegh["air_lbs"], k=5)
/var/folders/92/rrvsc6_x7slf490khsx_xn2c0000gn/T/ipykernel_59244/760923391.py:13: UserWarning: Numba not installed. Using slow pure python version.
  easyrsei["rsei_top"] = top_class(easyrsei["rsei_score"], k=4)


In [9]:
major_fac = (
    tri_allegh
    .merge(easyrsei, on="address", how="left")
    .assign(
        rsei_top=lambda d: d["rsei_top"].fillna(False).astype(bool),
        major=lambda d: d["tri_top"] | d["rsei_top"]
    )
    .query("major")
    .sort_values("air_lbs", ascending=False)
)

In [6]:
import geopandas as gpd
fac_gpd = gpd.GeoDataFrame(major_fac, geometry=gpd.points_from_xy(major_fac["lon"], major_fac["lat"]),
                       crs="EPSG:4326").to_crs("EPSG:2272")

## Step 3: Produce map of Allegheny County polluters 

In [13]:
alleg_outline = alleg.explore(fill=False)
fac_gpd.explore(column="tri_pounds", cmap="Reds", legend=True, marker_kwds={"radius": 5})

## Step 4: Calculate distance to nearest polluter for each parcel

## OLD: toxic tracker dashboard data processing

In [ ]:
tri_toxic_tracker = (pd.read_csv("easyRSI.csv"))
rsei_top10 = (
    tri_toxic_tracker
    .rename(
        columns={
            "TRI Facility Name": "tri_facility_name",
            "TRI Facility ID": "tri_facility_id",
            "Year": "year",
            "Air Releases (lb)": "air_releases_lb",
            "RSEI Hazard": "rsei_hazard"
            }
        )
    .sort_values('year', ascending=False)
    .drop_duplicates(subset=["tri_facility_id"], keep="first")
    .filter(items=["tri_facility_name", "tri_facility_id", "year", "rsei_hazard"])
    .assign(
        # convert rsei_hazard to numeric for analysis
        rsei_hazard=lambda x: pd.to_numeric(x["rsei_hazard"].str.replace(',', ''), errors='coerce')
    )
    .nlargest(10, 'rsei_hazard')
)
rsei_top10